# Data Engineering  - Loading/Cleaning/Pipeline
---

## 1 - Data Loading

In [6]:
import json
from pymongo import MongoClient

# connect to MongoDB and initialize client 
client = MongoClient('mongodb://localhost:27017/')
db = client['novacred_db']
collection = db['applications'] # define collection for more reliability  

# drop collection before loading multiple times
collection.drop()

# load JSON file
with open('../data/raw_credit_applications.json', 'r') as file:
    raw_data = json.load(file)

## 2 - Data Cleaning 

Following the project guidelines, we will address the six core data quality issues:
### 1. Duplicate records
MongoDB requires unique _id fields. Therefore, we check for duplicates.

In [7]:
unique_records = {}
duplicates_count = 0
# as "_id" is a unique identifier, documents with the same _id shall be deleted 
for application in raw_data:
    app_id = application['_id']
    
    # count duplicates
    if app_id in unique_records:
        duplicates_count += 1
        print(f"Duplicate found: {app_id}")
    
    # duplicates will be removed through overwriting
    unique_records[app_id] = application

clean_data = list(unique_records.values())

print(f"Found and removed {duplicates_count} duplicates.")

Duplicate found: app_042
Duplicate found: app_001
Found and removed 2 duplicates.


The duplicated documents are resubmitted applications with duplicate IDs. We will kept the most recent resubmission and load the unique records into the database.

In [8]:
# loading cleaned data 
collection.insert_many(clean_data)
print(f"Inserted {len(clean_data)} unique documents into MongoDB.")

Inserted 500 unique documents into MongoDB.


### 2. Inconsistent data types across records
Instead of guessing which fields have inconsistent data types, we will dynamically audit the entire collection. 
This script flattens every document and records the Python data types found for each field. If a field contains more than one data type across the dataset (e.g., both integers and strings), it will be flagged for cleaning.

In [9]:
from collections import defaultdict

# Dictionary to hold sets of data types for every field
field_types = defaultdict(set)

# Helper function to flatten nested JSON (e.g., turns {'financials': {'income': 50}} into 'financials.income': 50)
def flatten_doc(d, parent_key='', sep='.'):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_doc(v, new_key, sep=sep).items())
        elif isinstance(v, list):
            # For this audit, we skip complex arrays like spending_behavior
            pass 
        else:
            # We ignore null/None values for the type consistency check
            if v is not None and v != "":
                items.append((new_key, type(v).__name__))
    return dict(items)

# Scan every document in the database
for doc in collection.find({}):
    doc.pop('_id', None) # We don't need to check the MongoDB ID
    flat_doc = flatten_doc(doc)
    
    for field, data_type in flat_doc.items():
        field_types[field].add(data_type)

inconsistent_fields = {field: types for field, types in field_types.items() if len(types) > 1}

if not inconsistent_fields:
    print("No inconsistent data types found.")
else:
    for field, types in inconsistent_fields.items():
        print(f"FLAGGED: '{field}' contains mixed types: {types}")

--- INCONSISTENT DATA TYPE AUDIT ---
FLAGGED: 'financials.annual_income' contains mixed types: {'str', 'float', 'int'}


In [10]:
# audit flagged 'financials.annual_income' as having mixed string/int types
# We will dynamically find any strings in these numeric fields and cast them to integers

fields_to_fix = list(inconsistent_fields.keys())

for field in fields_to_fix:
    # Update query: Find documents where this field is a string, and cast it to an int
    result = collection.update_many(
        {field: {"$type": "string"}},
        [{"$set": {field: {"$toInt": f"${field}"}}}]
    )
    if result.modified_count > 0:
        print(f"Fixed: Casted {result.modified_count} string values to integers in '{field}'.")

Fixed: Casted 8 string values to integers in 'financials.annual_income'.

Data type consistency check complete.
